# 00 — Entorno A100 (canal-estoicismo)

Este notebook **prepara la sesión y no genera vídeo**. Si algo está mal, se ve aquí y no después de gastar unidades.

## Antes de empezar

1. `Entorno de ejecución > Cambiar tipo de entorno de ejecución > A100 GPU` (requiere Pro/Pro+).
2. Abre el panel de secretos (icono de la llave) y crea las claves que tengas:
   - `GEMINI_API_KEY` — recomendado, guiones largos de 5-8 min (gratis en https://aistudio.google.com/apikey)
   - `NVIDIA_API_KEY`, `OPENROUTER_API_KEY`, `HF_TOKEN` — opcionales, respaldo
3. Ejecuta las celdas **en orden**. Si la sesión se cae, vuelve a ejecutar solo este notebook.

## Qué se guarda y qué no

| ruta | dónde vive | sobrevive a la sesión |
|---|---|---|
| `output/*.mp4` y `*.story.json` | tu Drive | sí |
| `assets/images/*.jpg` (caché SDXL) | tu Drive | sí |
| `logs/` | tu Drive | sí |
| `temp/`, paquetes pip | /content (local) | no |

La caché de imágenes es lo que más unidades ahorra: si un reel se corta a medias, al repetirlo las imágenes ya generadas **no se vuelven a pagar**.


In [ ]:
#@title 1. Comprobar la GPU (debe decir A100) { display-mode: "form" }
import subprocess

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print(smi.stdout.strip() or smi.stderr.strip())

try:
    import torch
except ImportError as exc:
    raise SystemExit(f"torch no está disponible en este entorno: {exc}")

if not torch.cuda.is_available():
    raise SystemExit(
        "No hay GPU CUDA visible.\n"
        "Ve a Entorno de ejecución > Cambiar tipo de entorno de ejecución > A100 GPU,\n"
        "espera al reinicio y vuelve a ejecutar desde la celda 1."
    )

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
print(f"\nGPU   : {GPU_NAME}")
print(f"VRAM  : {VRAM_GB:.1f} GB")
print(f"torch : {torch.__version__}  |  CUDA {torch.version.cuda}")

if VRAM_GB < 30:
    print(f"\n[AVISO] {VRAM_GB:.0f} GB no es una A100 (40 GB).")
    print("        El perfil 'a100' (SDXL full + refiner + hires) necesita ~24 GB de VRAM.")
    print("        Si estás en una T4, usa --profile t4 en los notebooks siguientes.")
else:
    print("\n[OK] A100 lista.")


In [ ]:
#@title 2. Montar Google Drive (persistencia) { display-mode: "form" }
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/canal-estoicismo")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive listo:", DRIVE_ROOT)
existing = sorted(p.name for p in DRIVE_ROOT.iterdir())
print("Contenido:", existing or "(vacío)")


In [ ]:
#@title 3. Traer el repositorio { display-mode: "form" }
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/jorgeecort-boop/canal-estoicismo.git"
PROJECT_DIR = Path("/content/canal-estoicismo")

if (PROJECT_DIR / ".git").exists():
    code = subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"]).returncode
    print(f"\ngit pull -> {code}")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)

commit = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "log", "-1", "--oneline"],
    capture_output=True, text=True,
).stdout.strip()
print("commit:", commit)


In [ ]:
#@title 4. Cargar los helpers del proyecto { display-mode: "form" }
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/canal-estoicismo")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import scripts.colab_env as env

assert env.PROJECT_DIR.exists(), "falta el repositorio: ejecuta la celda 3"
print("proyecto:", env.PROJECT_DIR)
print("drive   :", env.DRIVE_ROOT)


In [ ]:
#@title 5. Enlazar output/ y la caché de imágenes a Drive { display-mode: "form" }
for line in env.link_persistence():
    print(" ", line)

# Comprobación real: escribir en el repo debe escribir en Drive.
probe = env.PROJECT_DIR / "output" / ".drive_check"
probe.parent.mkdir(parents=True, exist_ok=True)
probe.write_text("ok", encoding="utf-8")
assert (env.DRIVE_ROOT / "output" / ".drive_check").exists(), "el enlace a Drive no funciona"
probe.unlink()
print("\n[OK] output/, assets/images/ y logs/ escriben directamente en Drive.")


In [ ]:
#@title 6. Instalar dependencias (el torch de Colab NO se toca) { display-mode: "form" }
env.banner("Paquetes del sistema")
env.apt_install(env.SYSTEM_DEPS)

env.banner("Paquetes de Python")
env.pip_install(env.PYTHON_DEPS)

print("\n[OK] Instalación terminada.")
print("Si Colab pide reiniciar el entorno de ejecución, hazlo y vuelve a ejecutar desde la celda 1.")


In [ ]:
#@title 7. Verificar torch y las versiones clave { display-mode: "form" }
info = env.check_torch()
for key, value in info.items():
    print(f"  {key:<15} {value}")

env.gate(
    "CUDA disponible",
    bool(info.get("cuda_available")),
    "sin CUDA el perfil A100 no puede generar imágenes",
)

import importlib

for module in ("diffusers", "transformers", "accelerate", "moviepy", "edge_tts", "safetensors"):
    try:
        loaded = importlib.import_module(module)
        print(f"  {module:<13} {getattr(loaded, '__version__', 'ok')}")
    except Exception as exc:
        env.gate(f"import {module}", False, f"{type(exc).__name__}: {exc}")

print("\n[OK] Dependencias cargadas.")


In [ ]:
#@title 8. Cargar los secretos (API keys) { display-mode: "form" }
found = env.load_secrets()

if not (found.get("GEMINI_API_KEY") or found.get("GOOGLE_API_KEY")):
    print("\n[AVISO] Sin GEMINI_API_KEY el lote usará plantillas locales:")
    print("        guiones más cortos y menos variados. El pipeline funciona igual.")

print("\n[OK] Secretos cargados (los valores nunca se imprimen).")


In [ ]:
#@title 9. GATE — suite de tests completa { display-mode: "form" }
code = env.run([sys.executable, "-m", "pytest", "-q"], cwd=env.PROJECT_DIR)
env.gate(
    "suite de tests",
    code == 0,
    "no continúes hasta que la suite pase: es la garantía de que el código no está roto",
)


In [ ]:
#@title 10. GATE — doctor (entorno + perfil efectivo) { display-mode: "form" }
code = env.run(
    [
        sys.executable, "main.py", "doctor",
        "--format", "reel",
        "--profile", "auto",
        "--image-provider", "sdxl_local",
    ],
    cwd=env.PROJECT_DIR,
)
env.gate("doctor", code == 0, "revisa las líneas [FAIL] del doctor de arriba")


## Entorno listo

Si las dos últimas celdas dicen `[PASS]`, la sesión está preparada.

Siguiente paso: **`01_verificacion_A100.ipynb`** (comprueba de verdad SDXL full, la voz y la composición) y luego **`02_lote_reels.ipynb`**.

### Cómo ahorrar unidades

- Ejecuta la verificación **antes** de cada lote largo: si algo va a fallar, que falle en 2 minutos y no en 40.
- No borres `assets/images/` en Drive: es la caché que evita volver a pagar imágenes.
- Si un reel se corta, reanúdalo con `03_entrega.ipynb` usando el `.story.json` guardado.
- El `--profile auto` detecta la GPU sola: en A100 usa SDXL full + hires; en T4 vuelve a sdxl-turbo.
